# COMP7104 / DASC7104 — Practical Assignment 1: Analytical SQL with DuckDB

**Name:** Cao Junjia  **Student id:** 3036790730

Fill in the cells below. Keep the structure: one code cell per question with the query and its result. Run every cell before submitting (Kernel → Restart Kernel and Run All Cells), then export to PDF. The handout (PDF on Moodle) has the full question texts and the expected results.

In [1]:
import os, time, statistics, platform
import duckdb, pandas as pd

pd.set_option("display.max_rows", 200); pd.set_option("display.width", 160)
con = duckdb.connect("pa1.duckdb")

def show(sql):
    """Run a query and display its full result as a table."""
    return con.sql(sql).df()

def timed(sql, repeat=3):
    """Run a query once to warm up, then `repeat` times; return (rows, median ms)."""
    con.execute(sql).fetchall()
    ts = []
    for _ in range(repeat):
        t0 = time.perf_counter(); rows = con.execute(sql).fetchall(); ts.append(time.perf_counter() - t0)
    return rows, round(statistics.median(ts) * 1e3, 1)

print("DuckDB", duckdb.__version__)

DuckDB 1.5.6


## Part A — the movies dataset

In [2]:
con.execute("CREATE OR REPLACE TABLE movies AS SELECT * FROM 'movies.parquet'")
con.execute("CREATE OR REPLACE TABLE directors AS SELECT * FROM 'directors.parquet'")
show("SELECT (SELECT count(*) FROM movies) AS movies, (SELECT count(*) FROM directors) AS directors")   # 4773, 2349

,movies,directors
0,4773,2349


### A1 (10 points) — Directors called Martin: number of movies after 2000, 0 when none.

Expected: 8 rows, columns `id, name, nb_movies`.

In [3]:
A1 = """
SELECT d.id, d.name, count(m.id) AS nb_movies
FROM directors d
LEFT JOIN movies m ON m.director_id = d.id AND year(m.release_date::DATE) > 2000
WHERE d.name LIKE 'Martin %'
GROUP BY d.id, d.name
ORDER BY nb_movies DESC, d.name
"""
show(A1)

,id,name,nb_movies
0,4809,Martin Scorsese,7
1,4791,Martin Campbell,5
2,5886,Martin McDonagh,2
3,4983,Martin Brest,1
4,6328,Martin Koolhoven,1
5,6193,Martin Lawrence,1
6,5861,Martin Weisz,1
7,6039,Martin Ritt,0


### A2 (10 points) — Year-over-year change of total revenue, from 2000 on (CTE + LAG).

Expected: 18 rows, columns `y, total_revenue, previous_year, change_pct`.

In [4]:
A2 = """
WITH yearly AS (
  SELECT year(release_date::DATE) AS y, sum(revenue)::BIGINT AS total_revenue
  FROM movies
  GROUP BY y
),
lagged AS (
  SELECT y, total_revenue,
         lag(total_revenue) OVER (ORDER BY y) AS previous_year
  FROM yearly
)
SELECT y, total_revenue, previous_year,
       round(100.0 * (total_revenue - previous_year) / previous_year, 1) AS change_pct
FROM lagged
WHERE y >= 2000
ORDER BY y
"""
show(A2)

,y,total_revenue,previous_year,change_pct
0,2000,10952218337,10332740909,6.0
1,2001,13269869789,10952218337,21.2
2,2002,14609857556,13269869789,10.1
3,2003,14211646728,14609857556,-2.7
4,2004,16278686567,14211646728,14.5
5,2005,14931589218,16278686567,-8.3
6,2006,16635892750,14931589218,11.4
7,2007,16491621655,16635892750,-0.9
8,2008,18145379803,16491621655,10.0
9,2009,21072651506,18145379803,16.1


### A3 (10 points) — Christopher Nolan's career: cumulative revenue and revenue rank.

Expected: 8 rows, columns `release_date, original_title, revenue, cumulative_revenue, revenue_rank`.

In [5]:
A3 = """
SELECT m.release_date, m.original_title, m.revenue,
       (sum(m.revenue) OVER (ORDER BY m.release_date))::BIGINT AS cumulative_revenue,
       rank() OVER (ORDER BY m.revenue DESC) AS revenue_rank
FROM movies m JOIN directors d ON m.director_id = d.id
WHERE d.name = 'Christopher Nolan'
ORDER BY m.release_date
"""
show(A3)

,release_date,original_title,revenue,cumulative_revenue,revenue_rank
0,2000-10-11,Memento,39723096,39723096,8
1,2002-05-24,Insomnia,113714830,153437926,6
2,2005-06-10,Batman Begins,374218673,527656599,5
3,2006-10-19,The Prestige,109676311,637332910,7
4,2008-07-16,The Dark Knight,1004558444,1641891354,2
5,2010-07-14,Inception,825532764,2467424118,3
6,2012-07-16,The Dark Knight Rises,1084939099,3552363217,1
7,2014-11-05,Interstellar,675120017,4227483234,4


### A4 (10 points) — Top-3 directors per decade by total revenue, ties kept (RANK + QUALIFY).

Expected: 31 rows, columns `decade, name, total_revenue, nb_movies, rk`.

In [6]:
A4 = """
WITH per_decade AS (
  SELECT (year(m.release_date::DATE) // 10) * 10 AS decade,
         d.id, d.name,
         sum(m.revenue)::BIGINT AS total_revenue, count(*) AS nb_movies
  FROM movies m JOIN directors d ON m.director_id = d.id
  GROUP BY decade, d.id, d.name
)
SELECT decade, name, total_revenue, nb_movies,
       rank() OVER (PARTITION BY decade ORDER BY total_revenue DESC) AS rk
FROM per_decade
QUALIFY rk <= 3
ORDER BY decade, rk, name
"""
show(A4)

,decade,name,total_revenue,nb_movies,rk
0,1910,D.W. Griffith,8394751,1,1
1,1920,King Vidor,22000000,1,1
2,1920,Harry Beaumont,4358000,1,2
3,1920,Fritz Lang,650422,1,3
4,1930,Victor Fleming,433931426,2,1
5,1930,David Hand,184925486,1,2
6,1930,Frank Capra,21533101,3,3
7,1940,David Hand,350767150,2,1
8,1940,Ben Sharpsteen,84300000,1,2
9,1940,William Wyler,23650000,1,3


### A5 (10 points) — Share of movies directed by women per year from 2000 on (FILTER).

Expected: 18 rows, columns `y, nb_movies, by_women, pct_women`.

In [7]:
A5 = """
SELECT year(m.release_date::DATE) AS y,
       count(*) AS nb_movies,
       count(*) FILTER (d.gender = 1) AS by_women,
       round(100.0 * count(*) FILTER (d.gender = 1) / count(*))::INTEGER AS pct_women
FROM movies m JOIN directors d ON m.director_id = d.id
WHERE year(m.release_date::DATE) >= 2000
GROUP BY y
ORDER BY y DESC
"""
show(A5)

,y,nb_movies,by_women,pct_women
0,2017,1,0,0
1,2016,104,3,3
2,2015,210,11,5
3,2014,234,9,4
4,2013,230,6,3
5,2012,207,11,5
6,2011,223,13,6
7,2010,223,15,7
8,2009,245,16,7
9,2008,225,21,9


### A6 (10 points) — Movies and average known budget by decade and gender, with subtotals (ROLLUP).

Expected: 37 rows, columns `decade, gender, nb_movies, avg_budget`.

In [8]:
A6 = """
SELECT (year(m.release_date::DATE) // 10) * 10 AS decade,
       coalesce(d.gender::VARCHAR, 'all') AS gender,
       count(*) AS nb_movies,
       round(avg(m.budget) FILTER (m.budget > 0))::BIGINT AS avg_budget
FROM movies m JOIN directors d ON m.director_id = d.id
GROUP BY ROLLUP(decade, d.gender)
ORDER BY decade, grouping(d.gender), d.gender
"""
show(A6)

,decade,gender,nb_movies,avg_budget
0,1910,2,1,385907
1,1910,all,1,385907
2,1920,2,4,31081333
3,1920,all,4,31081333
4,1930,0,7,1663819
5,1930,2,8,1212347
6,1930,all,15,1438083
7,1940,0,7,1969286
8,1940,2,18,2296214
9,1940,all,25,2187238


### A7 (10 points) — The skyline of movies (NOT EXISTS over a CTE).

Expected: 36 rows, columns `id, original_title, revenue, budget, popularity, vote_average`.

In [9]:
A7 = """
WITH scored AS (
  SELECT id, original_title, revenue, budget, popularity,
         CASE WHEN vote_count > 10 THEN vote_average ELSE 0 END AS vote_average
  FROM movies
)
SELECT s.id, s.original_title, s.revenue, s.budget, s.popularity, s.vote_average
FROM scored s
WHERE NOT EXISTS (
  SELECT *
  FROM scored o
  WHERE o.popularity > s.popularity
    AND o.vote_average > s.vote_average
    AND o.revenue > s.revenue
    AND o.budget > s.budget
)
ORDER BY s.budget, s.id
"""
show(A7)

,id,original_title,revenue,budget,popularity,vote_average
0,47462,Whiplash,13092000,3300000,192,8.3
1,46934,The Godfather,245066411,6000000,143,8.4
2,46829,Pulp Fiction,213928762,8000000,121,8.3
3,46328,The Godfather: Part II,47542841,13000000,105,8.3
4,45891,千と千尋の神隠し,274925095,15000000,118,8.3
5,45587,The Empire Strikes Back,538400000,18000000,78,8.2
6,45415,Schindler's List,321365567,22000000,104,8.3
7,45584,ハウルの動く城,234710455,24000000,49,8.2
8,45478,The Shawshank Redemption,28341469,25000000,136,8.5
9,45844,もののけ姫,159375308,26500000,60,8.2


## Part B — under the hood

State your hardware here (edit this cell): **CPU** Apple M1 Pro · **cores** 8 (6 performance + 2 efficiency) · **RAM** 16 GB · **storage** SSD · **OS** macOS 26.6.2

The cell below prints what Python can see; complete the rest by hand.

In [10]:
print(platform.platform()); print(platform.processor() or platform.machine(), "|", os.cpu_count(), "logical cores")
try:
    print(round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 1), "GB RAM")
except (AttributeError, ValueError):
    pass

macOS-26.6.2-arm64-arm-64bit
arm | 8 logical cores
16.0 GB RAM


### Generate and load the scale-10 dataset (once; about ten seconds plus the load)

In [11]:
from make_dataset import ensure         # make_dataset.py from the setup archive, in this folder
ensure("pa1data", scale=10)             # generated once (~10 s); reused if already there at scale 10
for t in ("actor", "movie", "casting"):
    con.execute(f"CREATE OR REPLACE TABLE {t} AS SELECT * FROM 'pa1data/{t}.parquet'")
show("SELECT (SELECT count(*) FROM actor) AS actors, (SELECT count(*) FROM movie) AS movies, (SELECT count(*) FROM casting) AS castings")

using pa1data/ (scale 10: 9,999,744 castings)


,actors,movies,castings
0,2000000,1000000,9999744


### B1 (10 points) — three queries with timings

In [12]:
B1a = """
SELECT a.last_name
FROM actor a
JOIN casting c ON c.actor_id = a.actor_id
JOIN movie m ON m.movie_id = c.movie_id
WHERE a.first_name = 'Clint' AND m.year = 1995
"""
rows, ms = timed(B1a)
print(len(rows), "rows,", ms, "ms (median of 3)")
show(B1a + " LIMIT 20")

292 rows, 10.9 ms (median of 3)


,last_name
0,Coleman
1,Carpenter
2,Johnston
3,Silva
4,Nguyen
5,Crawford
6,Rogers
7,Carlson
8,Silva
9,Jenkins


In [13]:
B1b = """
SELECT (m.year // 10) * 10 AS decade,
       count(DISTINCT m.movie_id) AS nb_movies,
       round(count(*) / count(DISTINCT m.movie_id), 1) AS avg_cast_size
FROM movie m
JOIN casting c ON c.movie_id = m.movie_id
GROUP BY decade
ORDER BY decade
"""
rows, ms = timed(B1b)
print(len(rows), "rows,", ms, "ms (median of 3)")
show(B1b + " LIMIT 20")

13 rows, 132.8 ms (median of 3)


,decade,nb_movies,avg_cast_size
0,1900,14873,10.0
1,1910,32173,10.0
2,1920,45745,10.0
3,1930,56918,10.0
4,1940,67447,10.0
5,1950,76700,10.0
6,1960,85972,10.0
7,1970,95224,10.0
8,1980,103259,10.0
9,1990,111073,10.0


In [14]:
B1c = """
WITH am AS (
  SELECT c.actor_id, c.movie_id
  FROM casting c
  JOIN movie m ON m.movie_id = c.movie_id
  WHERE m.year >= 2000
)
SELECT a.first_name, a.last_name,
       count(DISTINCT am.movie_id) AS nb_movies,
       count(DISTINCT c2.actor_id) AS nb_coactors
FROM am
JOIN actor a ON a.actor_id = am.actor_id
JOIN casting c2 ON c2.movie_id = am.movie_id AND c2.actor_id <> am.actor_id
GROUP BY a.actor_id, a.first_name, a.last_name
ORDER BY nb_movies DESC, a.actor_id
LIMIT 10
"""
rows, ms = timed(B1c)
print(len(rows), "rows,", ms, "ms (median of 3)")
show(B1c)

10 rows, 2329.5 ms (median of 3)


,first_name,last_name,nb_movies,nb_coactors
0,Margaret,Coleman,4209,40622
1,David,Hudson,1509,14835
2,Michelle,Hill,1220,12170
3,Christopher,Kennedy,978,9630
4,Donald,Cox,779,7693
5,Stephanie,Austin,755,7307
6,Cynthia,Schmidt,702,6845
7,Brenda,Patterson,611,6008
8,Kenneth,Martinez,596,5903
9,Ronald,Hoffman,583,5738


Summarise here: 

| query | rows | median ms |
|---|---|---|
| (a) | 292 | 10.9 |
| (b) | 13 | 132.8 |
| (c) | 10 | 2329.5 |

### B2 (10 points) — plans, estimates and actuals

In [15]:
import html
from IPython.display import HTML, display
plan = con.execute("EXPLAIN " + B1a).fetchall()[0][1]
display(HTML("<pre style='font-size:8px; line-height:1.35'>" + html.escape(plan) + "</pre>"))

In [16]:
import html
from IPython.display import HTML, display
plan = con.execute("EXPLAIN ANALYZE " + B1a).fetchall()[0][1]
display(HTML("<pre style='font-size:8px; line-height:1.35'>" + html.escape(plan) + "</pre>"))

In [17]:
import html
from IPython.display import HTML, display
plan = con.execute("EXPLAIN " + B1c).fetchall()[0][1]
display(HTML("<pre style='font-size:8px; line-height:1.35'>" + html.escape(plan) + "</pre>"))

In [18]:
import html
from IPython.display import HTML, display
plan = con.execute("EXPLAIN ANALYZE " + B1c).fetchall()[0][1]
display(HTML("<pre style='font-size:8px; line-height:1.35'>" + html.escape(plan) + "</pre>"))

**Answers (about 200 words in total):**

1. Join order and probe side of the last join in (a): the optimizer joins `actor` (filtered to `first_name='Clint'`) with `casting` on `actor_id` first, then joins that intermediate result with `movie` on `movie_id`. In the last join the probe side is the actor⋈casting intermediate result (24,728 rows streamed through); the build side is `movie` filtered by `year=1995` (11,088 rows). This makes sense: the hash table is built on the smallest input, so it stays compact and cache-friendly, while the larger intermediate result is only streamed.
2. Filters pushed into scans, and the columns of `casting` actually read: `first_name='Clint'` is pushed into the `actor` scan and `year=1995` into the `movie` scan. At run time, dynamic filters are also pushed into the `casting` scan: min-max bounds (`actor_id>=397 AND actor_id<=1999689`, `movie_id>=111 AND movie_id<=999955`) and a Bloom filter (`actor_id IN BF(actor_id)`) derived from the build sides. Of `casting`, only the columns `actor_id` and `movie_id` are read; `role` and `billing` are pruned away.
3. Two operators, estimated vs actual rows: the `actor` scan is estimated at ~9,390 rows and actually outputs 5,037; the first hash join (`actor`⋈`casting`) is estimated at ~46,948 rows and actually outputs 24,728. The final join is estimated at ~320 vs 292 actual, which is accurate.
4. One estimate off by more than 2× and a hypothesis: the `casting` scan in (a) is estimated at 9,999,744 rows but emits only 114,133 (~88× off). The static estimate cannot foresee the dynamic filters (Bloom filter and zone maps) created at run time from the tiny build sides, so it assumes a full scan. In (c) the self-join is estimated at ~592.8M rows vs 31.1M actual (~19×): the optimizer assumes uniform, independent join keys, but the generator draws `actor_id` with a power law, so popular actors collide far more often than uniformity predicts.


### B3 (10 points) — one experiment: (i) storage format, (ii) data order, or (iii) parallelism

Keep the cells of the option you chose, delete the other two.

In [19]:
# (i) storage format: query (b) over native tables, Parquet files and CSV files
Q_native = B1b
Q_parquet = B1b.replace("FROM movie m", "FROM 'pa1data/movie.parquet' m").replace("JOIN casting c", "JOIN 'pa1data/casting.parquet' c")
Q_csv = B1b.replace("FROM movie m", "FROM read_csv('pa1data/movie.csv') m").replace("JOIN casting c", "JOIN read_csv('pa1data/casting.csv') c")
for label, q in (("native", Q_native), ("parquet", Q_parquet), ("csv", Q_csv)):
    print(label, timed(q)[1], "ms")

native 127.6 ms


parquet 127.1 ms


csv 430.2 ms


**Measurements:**

| setting | median ms |
|---|---|
| native tables | 127.6 |
| Parquet files | 127.1 |
| CSV files | 430.2 |

**Explanation (about 200 words):** the native tables and the Parquet files take essentially the same time (127.6 vs 127.1 ms), because DuckDB's native storage is itself a columnar, compressed format laid out in row groups with per-column statistics: loading the Parquet files into native tables is close to a byte-level copy, and at query time both are read the same way — only the three needed columns (`year`, `movie_id` from `movie`, `movie_id` from `casting`) are scanned, in compressed form, vector by vector. CSV is about 3.4× slower (430 ms) for three compounding reasons. First, parsing: every byte of the 350 MB of text must be scanned to find delimiters and newlines, and every value converted from text to an integer, whereas Parquet/native values are already binary. Second, size and compression: the CSV files are roughly 4× larger than the Parquet ones, so there is much more I/O and memory traffic, and values stay decompressed. Third, statistics: the CSV reader has no per-row-group min-max metadata, so nothing can ever be skipped, while the columnar formats carry zone maps. The residual difference between native and Parquet is small because the query is compute-bound on the join and aggregation, not on decoding.


## Declaration

Unused generative AI tools.
